In [2]:
!pip uninstall -y datasets transformers accelerate -q

!pip install -q \
    transformers==4.40.2 \
    accelerate==0.33.0 \
    datasets==2.18.0 \
    evaluate \
    jiwer \
    librosa

In [3]:
import transformers, datasets, accelerate
print(transformers.__version__)
print(datasets.__version__)
print(accelerate.__version__)

4.40.2
2.18.0
0.33.0


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import librosa

def prepare_batch(batch):
    audio, sr = librosa.load(batch["audio_path"], sr=16000)

    batch["input_features"] = processor.feature_extractor(
        audio,
        sampling_rate=16000
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

dataset = dataset.map(
    prepare_batch,
    remove_columns=dataset["train"].column_names,
    num_proc=1
)

print("Dataset Prepared Successfully ✅")

Map:   0%|          | 0/5346 [00:00<?, ? examples/s]

Map:   0%|          | 0/595 [00:00<?, ? examples/s]

Dataset Prepared Successfully ✅


In [3]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [1]:
!pip uninstall -y accelerate
!pip install accelerate==0.30.1

Found existing installation: accelerate 0.30.1
Uninstalling accelerate-0.30.1:
  Successfully uninstalled accelerate-0.30.1
  Using cached accelerate-0.30.1-py3-none-any.whl.metadata (18 kB)
Using cached accelerate-0.30.1-py3-none-any.whl (302 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.1 requires transformers, which is not installed.


In [3]:
!pip uninstall -y transformers accelerate datasets peft

In [1]:
!pip install transformers==4.40.2
!pip install datasets==2.18.0
!pip install accelerate==0.30.1
!pip install evaluate==0.4.2
!pip install jiwer

  Using cached transformers-4.40.2-py3-none-any.whl.metadata (137 kB)
Using cached transformers-4.40.2-py3-none-any.whl (9.0 MB)
  Using cached datasets-2.18.0-py3-none-any.whl.metadata (20 kB)
Using cached datasets-2.18.0-py3-none-any.whl (510 kB)
  Using cached accelerate-0.30.1-py3-none-any.whl.metadata (18 kB)
Using cached accelerate-0.30.1-py3-none-any.whl (302 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: evaluate
    Found existing installation: evaluate 0.4.6
    Uninstalling evaluate-0.4.6:
      Successfully uninstalled evaluate-0.4.6


In [2]:
import transformers, accelerate, datasets, evaluate

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)
print("Evaluate:", evaluate.__version__)

Transformers: 4.40.2
Accelerate: 0.30.1
Datasets: 2.18.0
Evaluate: 0.4.2


In [5]:
import os
import pandas as pd

BASE_PATH = "/content/drive/MyDrive/ASR"
CSV_PATH = os.path.join(BASE_PATH, "training_data_segmented.csv")

df = pd.read_csv(CSV_PATH)

AUDIO_FOLDER = "/content/drive/MyDrive/ASR/processed_training_data"

def fix_path(x):
    filename = x.split("\\")[-1]
    return os.path.join(AUDIO_FOLDER, filename)

df["audio_path"] = df["audio_path"].apply(fix_path)

print(df["audio_path"].iloc[0])
print("File exists:", os.path.exists(df["audio_path"].iloc[0]))
print("Total samples:", len(df))

/content/drive/MyDrive/ASR/processed_training_data/825780_seg_0.wav
File exists: True
Total samples: 5941


In [6]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(test_size=0.1)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['audio_path', 'text'],
        num_rows: 5346
    })
    test: Dataset({
        features: ['audio_path', 'text'],
        num_rows: 595
    })
})


In [7]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained("openai/whisper-small")

model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
)

# 🔥 CRITICAL FIX (prevents suppress_tokens error)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = None

model.to("cuda")

print("Model Loaded Successfully ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Model Loaded Successfully ✅


In [8]:
import librosa

def prepare_batch(batch):
    audio, sr = librosa.load(batch["audio_path"], sr=16000)

    batch["input_features"] = processor.feature_extractor(
        audio,
        sampling_rate=16000
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

dataset = dataset.map(
    prepare_batch,
    remove_columns=dataset["train"].column_names,
    num_proc=1
)

print("Dataset Prepared Successfully ✅")

Map:   0%|          | 0/5346 [00:00<?, ? examples/s]

Map:   0%|          | 0/595 [00:00<?, ? examples/s]

Dataset Prepared Successfully ✅


In [9]:
from dataclasses import dataclass
import torch

@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2Seq(processor)

print("Data Collator Ready ✅")

Data Collator Ready ✅


In [10]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

torch.cuda.empty_cache()

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/ASR/whisper_finetuned_v1",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=1e-5,
    warmup_steps=200,

    num_train_epochs=2,  # Balanced (no underfit, no overfit)

    fp16=True,
    gradient_checkpointing=True,

    save_strategy="no",
    evaluation_strategy="no",

    logging_steps=50,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    data_collator=data_collator,
)

print("Training Setup Ready ✅")

Training Setup Ready ✅


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [11]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...


Step,Training Loss
50,1.996500
100,1.129700
150,0.808300
200,0.680700
250,0.563500
300,0.431100
350,0.371100
400,0.318800
450,0.287100
500,0.290000


TrainOutput(global_step=668, training_loss=0.5853938435366054, metrics={'train_runtime': 3275.9244, 'train_samples_per_second': 3.264, 'train_steps_per_second': 0.204, 'total_flos': 3.08382358781952e+18, 'train_loss': 0.5853938435366054, 'epoch': 1.9985041136873598})

In [12]:
trainer.save_model("/content/drive/MyDrive/ASR/whisper_finetuned_v1/final")
processor.save_pretrained("/content/drive/MyDrive/ASR/whisper_finetuned_v1/final")

print("Model Saved Successfully ✅")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'begin_suppress_tokens': [220, 50257]}


Model Saved Successfully ✅
